In [ ]:
# hide
from collections.abc import Callable
from functools import cache

import numpy as np
import pyquist as pq
from scipy.signal import lfilter

In [ ]:
# collapse
SECONDS_PER_BEAT = 0.6                           # a tempo of 100 beats per minute
PITCH_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
CHORD_SHAPES = {"maj": [0, 4, 7], "min": [0, 3, 7], "7": [0, 4, 7, 10],
                "maj7": [0, 4, 7, 11], "min7": [0, 3, 7, 10]}   # semitones above the root


def pluck(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A Karplus-Strong plucked string (Chapter 12) that decays by 60 dB in `duration`
    seconds. An all-pass filter (Chapter 13) in the loop tunes it precisely."""
    P = f_s / pq.helper.pitch_to_frequency(pitch)   # the period, in (fractional) samples
    M = int(P - 0.6)                                 # the delay line (the low-pass adds 1/2 sample)
    D = P - 0.5 - M                                  # the all-pass adds the remaining D samples
    c = (1 - D) / (1 + D)                            # the all-pass gain for a delay of D samples
    g = 10 ** (-3 * P / (f_s * duration))            # decay by 60 dB over the duration
    x = np.zeros(int(duration * f_s))
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: a burst of white noise
    # The whole loop is one recursive filter (Chapter 9), which SciPy runs quickly:
    # y[n] = x[n] + c x[n-1] - c y[n-1] + (g/2) (c y[n-M] + (1 + c) y[n-M-1] + y[n-M-2])
    a = np.zeros(M + 3)
    a[:2] = [1.0, c]
    a[M:] -= 0.5 * g * np.array([c, 1 + c, 1.0])
    return pq.Audio(lfilter([1.0, c], a, x), f_s)


def piano(pitch: float, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    """A piano-like tone by additive synthesis (Chapter 3): six harmonics that decay
    exponentially, the higher ones faster, with a quick fade in and out to avoid clicks."""
    t = np.arange(int(duration * f_s)) / f_s
    f_0 = pq.helper.pitch_to_frequency(pitch)
    x = sum(np.sin(2 * np.pi * k * f_0 * t) * np.exp(-2 * k * t) / k for k in range(1, 7))
    fade = np.clip(np.minimum(t / 0.005, (duration - t) / 0.05), 0, 1)
    return pq.Audio(0.5 * x * fade, f_s)


@cache
def guitar(pitch: float, duration: float) -> pq.Audio:
    """A plucked string for the chords. To save time, each string is plucked just once
    (the first time it is needed), and then reused."""
    return 0.6 * pluck(pitch, duration)


def band(part: str, pitch: float, duration: float, **kwargs) -> pq.Audio:
    """The instrument for a lead sheet: a piano for the melody, a guitar for the chords."""
    return piano(pitch, duration) if part == "melody" else guitar(pitch, duration)


def lead_sheet_to_score(melody: list, chords: list, max_beats: float = 64) -> pq.Score:
    """A score, with times in seconds, of the first max_beats beats of a lead sheet. Each
    melody note lasts until the next one (at most two beats). Each chord is strummed, low
    to high, when it starts, and again on every other beat until the next chord."""
    events = []
    for i, (beat, pitch) in enumerate(melody):
        end = melody[i + 1][0] if i + 1 < len(melody) else beat + 1
        events.append((beat, {"part": "melody", "pitch": pitch,
                              "duration": min(end - beat, 2) * SECONDS_PER_BEAT}))
    for i, (beat, label) in enumerate(chords):
        root, family = label.split(":")
        bass = 48 + PITCH_NAMES.index(root)          # the root, between C3 and B3
        end = chords[i + 1][0] if i + 1 < len(chords) else beat + 4
        strums = [beat] + [b for b in range(int(beat) + 1, int(np.ceil(end))) if b % 2 == 0]
        for strum in strums:
            for j, interval in enumerate(CHORD_SHAPES[family]):
                events.append((strum + 0.03 * j, {"part": "chord", "pitch": bass + interval,
                                                  "duration": 2.0}))
    return pq.Score([(beat * SECONDS_PER_BEAT, kwargs) for beat, kwargs in events
                     if 0 <= beat < max_beats])

In [ ]:
# collapse
LOWEST, HIGHEST = 48, 84                         # the range of melody pitches, C3 to C6
VOCAB = (["<BOS>", "<EOS>", "BAR"]               # start of song, end of song, new bar
         + [f"CHORD_{root}:{kind}" for root in PITCH_NAMES for kind in CHORD_SHAPES]
         + [f"NOTE_{pitch}" for pitch in range(LOWEST, HIGHEST + 1)]
         + [f"SHIFT_{k}" for k in range(1, 16)])  # wait k sixteenth notes
INDEX = {token: i for i, token in enumerate(VOCAB)}


def tokens_to_events(tokens: list[int]) -> tuple[list, list]:
    """The melody ([beat, pitch] pairs) and chords ([beat, label] pairs) of a token
    sequence, up to the first <EOS>."""
    melody, chords, bar, position = [], [], -1, 0
    for token in tokens:
        kind, _, value = VOCAB[token].partition("_")
        if kind == "<EOS>":
            break
        elif kind == "BAR":
            bar, position = bar + 1, 0
        elif kind == "SHIFT":
            position += int(value)
        elif kind == "CHORD":
            chords.append([4 * bar + position / 4, value])
        elif kind == "NOTE":
            melody.append([4 * bar + position / 4, int(value)])
    return melody, chords


def load_model(path: str) -> Callable[[list[int]], np.ndarray]:
    """Load the pre-trained neural network f_theta, a small recurrent neural network (see
    notebooks/train-theorytab-rnn.ipynb). It maps the tokens so far to the probability
    of each possible next token. To save time, it remembers the tokens that it has
    already read (summarized in a vector h), so it only reads each new token once."""
    weights = np.load(path)
    assert list(weights["vocab"]) == VOCAB
    E, W_x, W_h, b_h, W_y, b_y = (weights[k] for k in ["E", "W_x", "W_h", "b_h", "W_y", "b_y"])
    state = {"seen": [], "h": np.zeros(len(b_h))}

    def f_theta(tokens: list[int]) -> np.ndarray:
        if tokens[:len(state["seen"])] != state["seen"]:       # a new sequence: start over
            state["seen"], state["h"] = [], np.zeros(len(b_h))
        h = state["h"]
        for token in tokens[len(state["seen"]):]:
            h = np.tanh(E[token] @ W_x + h @ W_h + b_h)
        state["seen"], state["h"] = list(tokens), h
        z = h @ W_y + b_y
        p = np.exp(z - z.max())                                # a softmax
        return p / p.sum()

    return f_theta

In [ ]:
f_theta = load_model("../assets/lead-sheet-rnn.npz")   # pre-trained on ~9,000 lead sheets
tokens = [INDEX["<BOS>"]]                               # the beginning of a song
while tokens.count(INDEX["BAR"]) <= 16:                 # generate 16 bars
    p = f_theta(tokens)                                 # P(next token | all tokens so far)
    p[INDEX["<EOS>"]] = 0                               # don't let the song end early
    tokens.append(np.random.choice(len(VOCAB), p=p / p.sum()))
print("Generated:", [VOCAB[token] for token in tokens[:16]], "...")
melody, chords = tokens_to_events(tokens)
pq.play(lead_sheet_to_score(melody, chords).render(band), normalize=True)